## 1. Setup — Imports and LLM Initialization


In [12]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.1 MB/s eta 0:00:00


In [23]:
!pip install -q sentence-transformers

In [3]:
import os
os.environ["GROQ_API_KEY"] = input("Enter your GROQ API Key: ")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.3,max_tokens=300)

## Step 2: Basic LLM Call


In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a concise data science tutor."),
    HumanMessage(content="Explain overfitting in 2 sentences.")
]

response = llm.invoke(messages)
print("=== LLM Response ===")
print(response.content)
print("\nResponse type:", type(response))
print("Model used:", response.response_metadata.get("model_name", "groq"))

=== LLM Response ===
Overfitting occurs when a model is too complex and learns the noise in the training data, resulting in poor performance on new, unseen data. This happens because the model has become specialized to the specific patterns and quirks of the training data, rather than generalizing to the underlying relationships and trends.

Response type: <class 'langchain_core.messages.ai.AIMessage'>
Model used: llama-3.3-70b-versatile


## Step 3: Prompt Templates

In [5]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} teacher. Give clear, practical explanations."),
    ("human", "Explain the concept of {concept} with a real-world analogy.")
])

filled_prompt = prompt_template.invoke({
    "domain": "machine learning",
    "concept": "gradient descent"
})

print("=== Filled Prompt ===")
for msg in filled_prompt.messages:
    print(f"[{msg.type.upper()}]: {msg.content}\n")

response = llm.invoke(filled_prompt)
print("=== LLM Response ===")
print(response.content)

=== Filled Prompt ===
[SYSTEM]: You are an expert machine learning teacher. Give clear, practical explanations.

[HUMAN]: Explain the concept of gradient descent with a real-world analogy.

=== LLM Response ===
Gradient descent is a fundamental concept in machine learning, and I'd be happy to explain it using a real-world analogy.

**Hiking Down a Mountain**

Imagine you're a hiker who wants to reach the bottom of a mountain. The mountain represents a complex mathematical function, and the bottom of the mountain represents the optimal solution (the lowest point).

You're currently standing at a random point on the mountain, and you want to find the shortest path to the bottom. The problem is, you can't see the entire mountain, and you don't know the exact path to take.

Here's where gradient descent comes in:

1. **Take a step**: You take a small step in a direction that you think will lead you downhill. This step represents a small change in the parameters of the mathematical function

## Step 4: Chains using LCEL (LangChain Expression Language)


In [6]:

summarize_prompt = ChatPromptTemplate.from_template(
    "Summarise the following text in exactly 3 bullet points:\n\n{text}"
)
parser = StrOutputParser()

summarize_chain = summarize_prompt | llm | parser

input_text = """
LangChain is an open-source framework for building applications powered by
large language models. It provides modular components like prompts, chains,
agents, and memory that can be combined to build complex AI pipelines.
It supports integrations with OpenAI, HuggingFace, vector databases like
FAISS and Pinecone, and tools like web search and Python execution.
"""

result = summarize_chain.invoke({"text": input_text})
print("=== Chain Output ===")
print(result)

translate_prompt = ChatPromptTemplate.from_template(
    "Translate this to Spanish:\n\n{summary}"
)
translate_chain = translate_prompt | llm | parser

full_chain = summarize_chain | (lambda x: {"summary": x}) | translate_chain
spanish_result = full_chain.invoke({"text": input_text})
print("\n=== Translated to Spanish ===")
print(spanish_result)

=== Chain Output ===
* LangChain is an open-source framework for building applications powered by large language models.
* The framework provides modular components such as prompts, chains, agents, and memory that can be combined to build complex AI pipelines.
* LangChain supports integrations with various tools and services, including OpenAI, HuggingFace, FAISS, Pinecone, web search, and Python execution.

=== Translated to Spanish ===
* LangChain es un marco de código abierto para construir aplicaciones impulsadas por grandes modelos de lenguaje.
* El marco proporciona componentes modulares, incluyendo instrucciones, cadenas, agentes y memoria, que se pueden combinar para construir tuberías de inteligencia artificial complejas.
* LangChain admite integraciones con diversas herramientas y servicios, como OpenAI, HuggingFace, FAISS, Pinecone, búsqueda en la web y ejecución de Python.


## Step 5: Memory


In [7]:
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use conversation history to give contextual answers."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = memory_prompt | llm

store = {}
def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

session = {"configurable": {"session_id": "student_001"}}

r1 = chain_with_memory.invoke({"input": "Hi! My name is Nikhil and I am studying machine learning."}, config=session)
print("Turn 1:", r1.content)

r2 = chain_with_memory.invoke({"input": "What topic am I studying?"}, config=session)
print("\nTurn 2:", r2.content)

r3 = chain_with_memory.invoke({"input": "What is my name?"}, config=session)
print("\nTurn 3:", r3.content)

print("\n=== Stored Message History ===")
for msg in store["student_001"].messages:
    print(f"[{msg.type}]: {msg.content[:80]}...")

Turn 1: Hello Nikhil, nice to meet you. Machine learning is a fascinating field, and I'd be happy to help you with any questions or topics you'd like to discuss. What specific areas of machine learning are you currently studying or interested in? Are you working on any projects or trying to learn a particular concept?

Turn 2: You're studying machine learning, Nikhil. Would you like to narrow it down to a specific area within machine learning, such as supervised learning, unsupervised learning, deep learning, or something else?

Turn 3: Your name is Nikhil.

=== Stored Message History ===
[human]: Hi! My name is Nikhil and I am studying machine learning....
[ai]: Hello Nikhil, nice to meet you. Machine learning is a fascinating field, and I'd...
[human]: What topic am I studying?...
[ai]: You're studying machine learning, Nikhil. Would you like to narrow it down to a ...
[human]: What is my name?...
[ai]: Your name is Nikhil....


## Step 6: Agents with Tools

In [42]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=791d7bbe777b65afc21ff1aa9362055004d29c73c1689a23630986be008d1b54
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [8]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.prebuilt import create_react_agent


wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500))
tools = [wiki_tool]


In [9]:
agent = create_react_agent(llm, tools)

result = agent.invoke({"messages": [{"role": "user", "content": "What is LangChain and when was it created?"}]})

print("=== Agent Reasoning Steps ===")
for msg in result["messages"]:
    print(f"[{msg.type.upper()}]: {msg.content[:200]}")
    print()

print("=== Final Answer ===")
print(result["messages"][-1].content)

/tmp/ipykernel_32483/2886898797.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


=== Agent Reasoning Steps ===
[HUMAN]: What is LangChain and when was it created?

[AI]: 

[TOOL]: Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain

[AI]: 

[TOOL]: Page: List of dates predicted for apocalyptic events
Summary: Predictions of apocalyptic events that will result in the extinction of humanity, a collapse of civilization, or the destruction of the pl

[AI]: Unfortunately, I was unable to find the creation date of LangChain using the available tools. Can I help you with anything else?

=== Final Answer ===
Unfortunately, I was unable to find the creation date of LangChain using the available tools. Can I help you with anything else?


## Step 7: Document Loaders + Vector Stores (RAG)

In [10]:
import os
os.environ["TQDM_NOTEBOOK"] = "0"

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

raw_documents = [
    Document(page_content="LangChain is an open-source framework for building LLM-powered applications. It was created by Harrison Chase in 2022.", metadata={"source": "langchain_intro.txt"}),
    Document(page_content="LCEL stands for LangChain Expression Language. It uses the pipe operator | to compose chains. Example: chain = prompt | llm | parser", metadata={"source": "lcel_guide.txt"}),
    Document(page_content="Agents in LangChain follow the ReAct pattern: the LLM reasons about which tool to use, calls it, observes the result, and repeats.", metadata={"source": "agents_guide.txt"}),
    Document(page_content="Vector stores like FAISS store document embeddings and enable semantic similarity search. They are the core of RAG pipelines.", metadata={"source": "vectorstores.txt"}),
    Document(page_content="Memory in LangChain persists conversation history. RunnableWithMessageHistory wraps any chain to add stateful memory.", metadata={"source": "memory_guide.txt"}),
    Document(page_content="LangChain supports many LLM providers: OpenAI GPT-4, Anthropic Claude, Google Gemini, Meta LLaMA via Groq, and local models via Ollama.", metadata={"source": "providers.txt"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks   = splitter.split_documents(raw_documents)
print(f"✓ Split into {len(chunks)} chunks")


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2",model_kwargs={"device": "cpu"},show_progress=False)
vectordb   = FAISS.from_documents(chunks, embeddings)
print(f"✓ Stored in FAISS vector store")

query   = "How does memory work in LangChain?"
results = vectordb.similarity_search(query, k=2)
print(f"\n=== Top 2 chunks for: '{query}' ===")
for i, doc in enumerate(results):
    print(f"\nChunk {i+1} (source: {doc.metadata['source']}):")
    print(doc.page_content)

✓ Split into 6 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Stored in FAISS vector store

=== Top 2 chunks for: 'How does memory work in LangChain?' ===

Chunk 1 (source: memory_guide.txt):
Memory in LangChain persists conversation history. RunnableWithMessageHistory wraps any chain to add stateful memory.

Chunk 2 (source: agents_guide.txt):
Agents in LangChain follow the ReAct pattern: the LLM reasons about which tool to use, calls it, observes the result, and repeats.


In [13]:
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the context provided.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

questions = [
    "What is LCEL and how does it work?",
    "Which LLM providers does LangChain support?",
    "How do agents decide which tool to use?"
]

print("=== RAG Pipeline Results ===\n")
for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-" * 50)

=== RAG Pipeline Results ===

Q: What is LCEL and how does it work?
A: LCEL stands for LangChain Expression Language. It uses the pipe operator | to compose chains, allowing for the creation of sequences such as "prompt | llm | parser".
--------------------------------------------------
Q: Which LLM providers does LangChain support?
A: LangChain supports the following LLM providers: OpenAI GPT-4, Anthropic Claude, Google Gemini, Meta LLaMA via Groq, and local models via Ollama.
--------------------------------------------------
Q: How do agents decide which tool to use?
A: Agents decide which tool to use by reasoning about it, following the ReAct pattern. The LLM (Large Language Model) reasons about which tool to use.
--------------------------------------------------


## Summary
LangChain has 8 core building blocks:
1. **LLM** — the brain, generates text (we used Groq + LLaMA)
2. **Prompt Template** — structures what we send to the LLM
3. **Chain** — connects prompt then LLM then output in one pipeline
4. **Memory** — makes the LLM remember past messages
5. **Agent** — lets the LLM decide which tool to use
6. **Tool** — external function the agent can call (e.g. Wikipedia)
7. **Vector Store** — stores documents for semantic search (FAISS)
8. **RAG** — retrieves relevant docs and feeds them to the LLM

All 8 components were implemented and tested above.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import nbformat, shutil

path = "/content/drive/MyDrive/Colab Notebooks/GenAI_Task_2_Langchain.ipynb"

shutil.copy(path, path.replace(".ipynb", "_backup.ipynb"))

with open(path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

nb.metadata.pop("widgets", None)

for cell in nb.cells:
    cell.get("metadata", {}).pop("widgets", None)

with open(path, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Done")

Done
